# Official Benchmark: Merge & vLLM Evaluation
This notebook completes the 3-step pipeline to run the massive 30,000 image benchmark while carefully managing Kaggle's strict 20GB `/kaggle/working` disk limit.

In [1]:
# Cell 1a: The Merge Install
!pip install -q transformers==4.49.0 peft==0.14.0 accelerate==1.2.1 huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 559.2 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 25.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.8/374.8 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.4/336.4 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 95.8 MB/s eta 0:00:00


## Step A: Merge the Weights & Push to HF
vLLM cannot load a 4-bit base model + LoRA. We must load the base model in fp16, attach the LoRA, and fuse them. 
**Disk Management:** We save the massive 15GB merged model to `/tmp` (which has ~70GB) instead of `/kaggle/working` to avoid disk crashes. We then explicitly push it to Hugging Face.

In [3]:
import torch
import gc
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from peft import PeftModel
from kaggle_secrets import UserSecretsClient

# Extract token early for private repository access
hf_token = UserSecretsClient().get_secret("HF_TOKEN")

# 1. Load full precision base model (FP16)
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct", 
    torch_dtype=torch.float16, 
    device_map="cpu" # Load to CPU first to avoid OOM
)
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")

# 2. Attach LoRA (using token to access private repo)
model = PeftModel.from_pretrained(
    base_model, 
    "AbrarAlam/disasterm3-qwen25vl7b-qlora", 
    subfolder="checkpoints/checkpoint-102",
    token=hf_token
)

# 3. Merge (Result is an FP16 model) and save to /tmp
print("Merging weights...")
merged_model = model.merge_and_unload()
MERGED_DIR = "/tmp/disasterm3_merged"
merged_model.save_pretrained(MERGED_DIR)
processor.save_pretrained(MERGED_DIR)
print(f"Merged model saved to {MERGED_DIR}!")

# 4. Push to HF (CRITICAL for backup and future-proofing)
try:
    hf_repo_id = "AbrarAlam/disasterm3-qwen2.5vl7b-mergedFP" # Appending FP to denote it's unquantized
    print(f"Pushing to Hugging Face: {hf_repo_id}...")
    merged_model.push_to_hub(hf_repo_id, token=hf_token, private=True)
    processor.push_to_hub(hf_repo_id, token=hf_token, private=True)
    print("Successfully pushed to Hugging Face!")
except Exception as e:
    print(f"Could not push to HF (make sure HF_TOKEN is in secrets): {e}")

# 5. Aggressive Memory Cleanup (Mandatory before starting vLLM)
del base_model
del model
del merged_model
gc.collect()
torch.cuda.empty_cache()
print("RAM and VRAM cleared for vLLM.")


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00005.safetensors:   0%|          | 0.00/3.90G [00:00<?, ?B/s]

model-00002-of-00005.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00003-of-00005.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

...

No files have been modified since last commit. Skipping to prevent empty commit.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Successfully pushed to Hugging Face!
RAM and VRAM cleared for vLLM.
